# 04 — Simulator model selection (E4)
Results §4–6, Methods selection. **LOAD+VERIFY** (v6 PCA canonical) + **RECOMPUTE** E4.13.

**Provenance.** Built by `~/.claude/skills/repro-notebook/SKILL.md` (Phase 3).
Companion docs: `docs/PAPER/repro/{MANIFEST.md, MAP.md, REPORT.md}`.
Target paper: `docs/PAPER/main.tex` → `Results/results_v4.tex`.
Helpers: `docs/PAPER/repro/_repro_util.py`.

**Modes.** `RECOMPUTE` = computed locally from C010 amplitudes. `LOAD+VERIFY` =
read the committed result JSON and compare to the printed paper value. SRM/BrainIAK
numbers are read from committed JSON (no MPI in this kernel).

**Source & code map**

| id | reported | source JSON (`future_phase2_filter_optimization/results/`) | mode |
|---|---|---|---|
| E4.4 | deutan argmin (6,−42), L_test −2.36 (IQR 2.15) | `s10b…sub-08.json` `summary['γOY|RDMV2|noLOCO'].per_model.2comp` | LOAD+VERIFY |
| E4.5 | protan argmin (2,+24), L_test −1.54 (IQR 1.42) | `…sub-09.json` `summary['γALL|RDMV1|noLOCO']…` | LOAD+VERIFY |
| E4.11 | N=300 | s10b `meta.N_resamples` | LOAD |
| E4.13 | pre-image 26.3 / 16.2 | `exp2_preimage/sub-{08,09}_…json` | RECOMPUTE |

Note: the gamma-variant combo key matters — `γOY` (deutan) / `γALL` (protan), not the
generic `γ_` atom; using the wrong key gives −2.14/−2.12.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))   # docs/PAPER/repro
import numpy as np
import _repro_util as U
U._RESULTS.clear()   # fresh check log per notebook
print("repo:", U.REPO)

repo: /Users/jinilkim/Library/CloudStorage/OneDrive-Personal/Projects/colorBlind_analysis


### E4.4 / E4.5 — selected 2-component argmin + held-out test-loss

In [2]:
def twocomp(sub, combo):
    j = U.load_json(U.P2 / f"results/s10_inclusion/s10b_v6_pca_rdm_results_{sub}.json")
    pm = j["summary"][combo]["per_model"]["2comp"]
    ps = pm.get("param_summary", {})
    return j, pm, ps
# production argmin (beta_s, beta_c) from the held-out LOO fit (s18 phase_b_fit)
s18 = U.load_json(U.P2 / "results/s10_inclusion/s18_heldout_predictive.json")["candidates"]
fit = {c["id"]: c["phase_b_fit"] for c in s18}
U.check("E4.4 deutan beta_s", fit["S08-robust"]["beta_s"], 6.0); U.check("E4.4 deutan beta_c", fit["S08-robust"]["beta_c"], -42.0)
U.check("E4.5 protan beta_s", fit["S09-primary"]["beta_s"], 2.0); U.check("E4.5 protan beta_c", fit["S09-primary"]["beta_c"], 24.0)
# production combos (gamma variant matters: deutan=gammaOY, protan=gammaALL)
j8, pm8, ps8 = twocomp("sub-08", "γOY|RDMV2|noLOCO")
j9, pm9, ps9 = twocomp("sub-09", "γALL|RDMV1|noLOCO")
U.check("E4.11 N_resamples", j8["meta"]["N_resamples"], 300)
U.check("E4.4 deutan L_test", pm8["test_loss_median"], -2.36, tol=0.01); U.check("E4.4 deutan L_test IQR", pm8["test_loss_iqr"], 2.15, tol=0.01)
U.check("E4.5 protan L_test", pm9["test_loss_median"], -1.54, tol=0.01); U.check("E4.5 protan L_test IQR", pm9["test_loss_iqr"], 1.42, tol=0.01)
print("  (gamma variant is significant: deutan=gammaOY, protan=gammaALL; the generic 'gamma_' key is a different atom and gives -2.14/-2.12.)")

[OK ] E4.4 deutan beta_s: produced=6.0  reported=6.0
[OK ] E4.4 deutan beta_c: produced=-42.0  reported=-42.0
[OK ] E4.5 protan beta_s: produced=2.0  reported=2.0
[OK ] E4.5 protan beta_c: produced=24.0  reported=24.0
[OK ] E4.11 N_resamples: produced=300  reported=300
[OK ] E4.4 deutan L_test: produced=-2.359316295724163  reported=-2.36
[OK ] E4.4 deutan L_test IQR: produced=2.149687623669479  reported=2.15
[OK ] E4.5 protan L_test: produced=-1.5390701698772489  reported=-1.54
[OK ] E4.5 protan L_test IQR: produced=1.416685834164569  reported=1.42
  (gamma variant is significant: deutan=gammaOY, protan=gammaALL; the generic 'gamma_' key is a different atom and gives -2.14/-2.12.)


### E4.13 — filter pre-image mean |δθ| (RECOMPUTE from committed pre-image)

In [3]:
U.check("E4.13 deutan mean|dtheta|", U.preimage_mean_abs_delta("08"), 26.3, tol=0.1)
U.check("E4.13 protan mean|dtheta|", U.preimage_mean_abs_delta("09"), 16.2, tol=0.1)

[OK ] E4.13 deutan mean|dtheta|: produced=26.283969800549002  reported=26.3
[OK ] E4.13 protan mean|dtheta|: produced=16.203372945287562  reported=16.2


In [4]:
U.summary()


=== 11/11 checks reproduced ===
